# Cuaderno 13 — Pruebas finales, bajo los protocolos declarados

**Antes de abrir este cuaderno, lo que no está aquí y por qué.** No hay XGBoost, ni redes
neuronales, ni ajuste de hiperparámetros, ni remuestreo de la clase minoritaria. No es por falta
de tiempo: es porque con 15 conglomerados positivos la amplitud del intervalo del estadístico es
0,442 (cuaderno 12), de modo que **cualquier diferencia que produzca un algoritmo nuevo se mueve
dentro del ruido y no se puede distinguir de la suerte.** Y probar cinco algoritmos para quedarse
con el mejor es exactamente lo que midió la entrada F16: elegir el mejor de cuatro especificaciones
infló la cifra 0,108 sin comprar nada fuera de muestra.

Si en la defensa preguntan por qué no se probaron más modelos, la respuesta defendible es que se
midió el techo de precisión del dato y que por debajo de ese techo la comparación entre algoritmos
no es informativa. Esa respuesta está sostenida por la curva del paso 7 del cuaderno 12.

**Lo que sí hace este cuaderno son cinco pruebas que o agregan información, o reducen varianza sin
selección, o verifican algo que todavía no se había verificado.** Van de la más barata a la más
cara, y en ese orden, porque la primera decide si vale la pena gastar tiempo en las demás.

| Paso | Qué responde | Costo |
| --- | --- | --- |
| 1 | ¿El ruido viene del bagging o de la partición? Decide si subir las bolsas sirve | ~15 min |
| 2 | ¿Qué pasa al restringir al universo de competencia? | ~15 min |
| 3 | ¿El conjunto reducido por bloques pierde algo? | ~15 min |
| 4 | ¿Cuántos aciertos hay en el tope de la lista, que es lo que usaría una entidad? | ~3 min |
| 5 | Permutaciones a 199 para poder afirmar p ≤ 0,005 | horas |

El paso 5 es caro de verdad y la celda te cronometra una permutación antes de empezar para que
decidas con un número en la mano, no a ciegas.


## Paso 0 · Entorno

El protocolo canónico, copiado del paso 7b del cuaderno 8 y sin un solo número cambiado: 25
bolsas, 5 semillas, muestra de comparación de 600 no etiquetados por año con semilla 777. Si
alguno de esos cuatro cambiara, nada de lo que mida este cuaderno sería comparable con lo que ya
está reportado.

La única diferencia respecto al cuaderno 8 es la función `evaluacion`, que devuelve el cuadro
completo en lugar de solo la mediana. Hace falta para el paso 4, y el estadístico que calcula es
idéntico.

Se trabaja sobre `final_panel.parquet`, que es la tabla canónica que dejó el cuaderno 12.


In [ ]:
# ============ Paso 0 · Entorno · protocolo canonico, identico al paso 7b ============
import os, sys, json, time, io, contextlib, hashlib, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
from sklearn.ensemble import HistGradientBoostingClassifier
os.chdir(r'C:\\SERCOP_FINAL')
R = '03_resultados'; OUT, OUT9, OUTF = f'{R}/c8_', f'{R}/c9_', f'{R}/final_'
SEMILLA = 20261001

# Los cuatro numeros del protocolo. NO se tocan en este cuaderno: si cambian, nada de lo que
# se mida aqui es comparable con el cuaderno 8.
K_EV, R_SEM, N_MUESTRA, SEM_MUESTRA, RATIO_U = 25, 5, 600, 777, 5

def muestra_comparacion(y, anios, semilla=SEM_MUESTRA, n_por_anio=N_MUESTRA):
    r = np.random.default_rng(semilla); partes = []
    for a in np.unique(anios):
        m = np.where((y == 0) & (anios == a))[0]
        if len(m): partes.append(r.choice(m, size=min(n_por_anio, len(m)), replace=False))
    return np.concatenate(partes)

def pu_bag(Xp, Xu, Xs, k, seed, ratio=RATIO_U):
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    with contextlib.redirect_stderr(io.StringIO()):
        for i in range(k):
            idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
            Xtr = np.vstack([Xp, Xu[idx]]); ytr = np.r_[np.ones(n), np.zeros(len(idx))]
            util = np.array([np.unique(Xtr[~np.isnan(Xtr[:,j]), j]).size >= 2
                             for j in range(Xtr.shape[1])])
            if util.sum() == 0: continue
            m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                    min_samples_leaf=5, l2_regularization=1.0, random_state=i).fit(Xtr[:,util], ytr)
            acum += m.predict_proba(Xs[:,util])[:,1]; hechos += 1
    assert hechos > 0
    return acum / hechos

def evaluacion(X, y, gi, anios, n_folds, k, semilla, Xu=None):
    """Devuelve el cuadro de evaluacion completo, no solo el estadistico: hace falta para
    medir precision en el tope (paso 4) y no solo la mediana."""
    r = np.random.default_rng(semilla); gp = np.unique(gi[y == 1])
    if len(gp) < 2: return None
    idx_s = np.concatenate([np.where(y == 1)[0], muestra_comparacion(y, anios)])
    Xs = X[idx_s]
    if Xu is None: Xu = X[y == 0]
    score = np.full(len(idx_s), np.nan); acum = np.zeros(len(idx_s)); nf = 0
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(r.permutation(gp), min(n_folds, len(gp))) if len(f)])
    for j, fuera in enumerate(folds):
        dentro = np.isin(gi, fuera)
        s = pu_bag(X[(y == 1) & ~dentro], Xu, Xs, k, semilla + j)
        donde = dentro[idx_s] & (y[idx_s] == 1)
        score[donde] = s[donde]; acum += s; nf += 1
    falta = np.isnan(score); score[falta] = (acum / nf)[falta]
    ev = pd.DataFrame({'y': y[idx_s], 'anio': anios[idx_s], 'm': score})
    ev['pct'] = ev.groupby('anio')['m'].rank(pct=True)
    return ev

def estadistico(X, y, gi, anios, n_folds, k, semilla, Xu=None):
    ev = evaluacion(X, y, gi, anios, n_folds, k, semilla, Xu)
    return np.nan if ev is None else float(np.nanmedian(ev.pct.values[ev.y.values == 1]))

def corrida(d, cy, cg, cols, n_folds, k=K_EV, semillas=R_SEM, etq=''):
    """Media y recorrido sobre varias semillas. Es la cifra que va a la tesis."""
    y = d[cy].values.astype(int); gi = pd.factorize(d[cg].values)[0]
    an = d.anio_proceso.values;   X = d[cols].astype(float).values
    Xu = X[y == 0]
    v = np.array([estadistico(X, y, gi, an, n_folds, k, SEMILLA + 31*s, Xu=Xu)
                  for s in range(semillas)])
    v = v[np.isfinite(v)]
    return {'caso': etq, 'n_vars': len(cols), 'filas': len(d), 'positivos': int(y.sum()),
            'conglomerados': int(pd.Series(gi[y == 1]).nunique()),
            'media': round(float(v.mean()), 3), 'min': round(float(v.min()), 3),
            'max': round(float(v.max()), 3), 'recorrido': round(float(v.max()-v.min()), 3),
            'n_semillas': len(v)}

# --- la base: se usa final_panel, que es la canonica del cuaderno 12 ---
ruta = f'{OUTF}panel.parquet' if os.path.exists(f'{OUTF}panel.parquet') else f'{OUT}panel.parquet'
P = pd.read_parquet(ruta)
print(f'panel: {os.path.basename(ruta)} | {P.shape[0]:,} filas x {P.shape[1]} columnas')
if 'rep_norm' not in P.columns:
    P = P.merge(pd.read_parquet(f'{OUT}supercias.parquet')[['proveedor_ruc','rep_norm']],
                on='proveedor_ruc', how='left')
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc.astype(str))
with open(f'{OUT}vars.json', encoding='utf-8') as f: VARS = json.load(f)
VARS = {k: [c for c in v if c in P.columns] for k, v in VARS.items()}
print('conjuntos:', {k: len(v) for k, v in VARS.items()})
uA = P.aparece_oferentes.values.astype(bool)
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
print(f'universo A (aparece en oferentes): {uA.sum():,} | universo B: {uB.sum():,}')
print(f'\nprotocolo: K={K_EV} bolsas, {R_SEM} semillas, muestra {N_MUESTRA}/anio, semilla {SEM_MUESTRA}')

## Paso 1 · De dónde viene el ruido

El recorrido entre semillas de A completo es 0,034. Antes de gastar tiempo en cualquier otra
cosa hay que saber de dónde sale, porque la respuesta decide si subir las bolsas sirve.

Hay dos fuentes posibles y se distinguen fácil. Si el ruido viene del **bagging** —cada bolsa
sortea no etiquetados distintos— entonces subir K lo promedia y desaparece: es precisión gratis.
Si viene de la **partición de pliegues y de la muestra de comparación**, subir K no lo toca y
sería tiempo tirado.

Una advertencia que vale repetir: esto mide el ruido del **procedimiento**, no el de la muestra.
El ruido de tener solo 15 conglomerados no lo mueve ningún K, y es un orden de magnitud mayor.


In [ ]:
# ============ Paso 1 · De donde viene el ruido: bolsas o particion ============
# El recorrido entre semillas de A completo es 0,034. Antes de gastar tiempo en nada mas hay
# que saber de donde sale, porque la respuesta decide si subir las bolsas sirve o es tiempo
# tirado. Dos fuentes posibles:
#   - el bagging: cada bolsa sortea no etiquetados distintos. Subir K lo promedia y se va.
#   - la particion de pliegues y la muestra de comparacion: subir K NO lo toca.
# Si el recorrido baja con K, hay precision gratis. Si no baja, K ya esta saturado.
K_PRUEBA, SEM_PRUEBA = [10, 25, 60], 3

d = P; cols = VARS['A']
fil, t0 = [], time.time()
for k in K_PRUEBA:
    r = corrida(d, 'y_fantasma', 'proveedor_ruc', cols, None, k=k, semillas=SEM_PRUEBA,
                etq=f'A completo, K={k}')
    r['segundos'] = round(time.time() - t0)
    fil.append(r)
    print(f"  K={k:3d}  media {r['media']:.3f}  recorrido {r['recorrido']:.3f}   "
          f"({r['segundos']} s acumulados)")

T1 = pd.DataFrame(fil)[['caso','media','min','max','recorrido','n_semillas','segundos']]
print('\n' + T1.to_string(index=False))
T1.to_csv(f'{OUT}ruido_bolsas.csv', index=False)

rec = T1.recorrido.values
if len(rec) >= 2 and rec[-1] < rec[0] * 0.6:
    print(f'\nEl recorrido cae de {rec[0]:.3f} a {rec[-1]:.3f} al subir las bolsas: una parte')
    print('importante del ruido es del bagging y subir K compra precision sin costo metodologico.')
    print('Conviene correr las cifras finales con el K mas alto que el tiempo permita, y')
    print('DECLARARLO, porque cambia el protocolo respecto al cuaderno 8.')
else:
    print(f'\nEl recorrido va de {rec[0]:.3f} a {rec[-1]:.3f}: subir las bolsas NO lo baja.')
    print('El ruido viene de la particion de pliegues y de la muestra de comparacion, no del')
    print('bagging. Subir K seria tiempo tirado y el protocolo del cuaderno 8 se queda como esta.')
print('\nOjo: esto mide el ruido del PROCEDIMIENTO, no el de la muestra. El ruido de tener solo')
print('15 conglomerados (amplitud 0,442, cuaderno 12) no lo mueve ningun K.')

## Paso 2 · Restricción al universo de competencia

Esta no es una prueba de robustez cualquiera. El cuaderno 12 midió que los 66.110 proveedor-año
que **solo** compran por catálogo electrónico no contienen ni un positivo: son el 28,4 % del
panel y no pueden aportar señal, solo ruido a la población de comparación.

Quitarlos es una decisión declarada de antemano y con motivo medido, no una búsqueda de un número
mejor. Y por eso mismo hay que comprometerse con el resultado antes de verlo: **si la cifra sube,
quitar el catálogo quitó ruido; si baja, el catálogo aportaba contraste útil y hay que
declararlo.** Las dos lecturas son publicables. Lo que no sería publicable es correr las dos
versiones y quedarse con la que convenga.


In [ ]:
# ============ Paso 2 · Restriccion al universo de competencia ============
# Esta no es una prueba de robustez cualquiera: el cuaderno 12 midio que los 66.110
# proveedor-ano que SOLO compran por catalogo no contienen ni un positivo. Son el 28,4 % del
# panel y no pueden aportar senal, solo ruido a la poblacion de comparacion. Quitarlos es una
# decision declarada de antemano y con motivo medido, no una busqueda de un numero mejor.
try:
    pr = pd.read_parquet(f'{OUT9}procedimiento.parquet', columns=['ocid','es_catalogo'])
    base = f'{OUTF}procesos.parquet' if os.path.exists(f'{OUTF}procesos.parquet') else f'{R}/base_maestra_v4.parquet'
    pv = pd.read_parquet(base, columns=['ocid','proveedor_ruc','anio_proceso']).merge(pr, on='ocid', how='left')
    frac = (pv.groupby(['proveedor_ruc','anio_proceso'])['es_catalogo'].mean()
              .rename('frac_catalogo').reset_index())
    P2 = P.merge(frac, on=['proveedor_ruc','anio_proceso'], how='left')
    solo_cat = (P2.frac_catalogo >= 0.999).fillna(False).values
    print(f'proveedor-ano solo catalogo: {solo_cat.sum():,} de {len(P2):,} '
          f'({solo_cat.mean()*100:.1f} %)')
    print(f'positivos dentro de ese grupo: {int(P2.loc[solo_cat, "y_fantasma"].sum())}')
    comp = P2[~solo_cat].copy()
    print(f'universo de competencia: {len(comp):,} filas | '
          f'{int(comp.y_fantasma.sum())} positivos en '
          f'{comp.loc[comp.y_fantasma==1,"proveedor_ruc"].nunique()} conglomerados')

    fil, t0 = [], time.time()
    cA = [c for c in VARS['A'] if c in comp.columns]
    fil.append(corrida(comp, 'y_fantasma', 'proveedor_ruc', cA, None,
                       etq='A completo · universo de competencia'))
    print(f"  {fil[-1]['caso']:42s} {fil[-1]['media']:.3f} "
          f"[{fil[-1]['min']:.3f}-{fil[-1]['max']:.3f}]  ({time.time()-t0:.0f} s)")
    uBc = (comp.tiene_grupo & comp.aparece_oferentes).values.astype(bool)
    cB = [c for c in VARS['B'] if c in comp.columns]
    if uBc.sum() > 1000:
        fil.append(corrida(comp[uBc], 'y_intragrupo', 'grupo_B', cB, 5,
                           etq='B completo · universo de competencia'))
        print(f"  {fil[-1]['caso']:42s} {fil[-1]['media']:.3f} "
              f"[{fil[-1]['min']:.3f}-{fil[-1]['max']:.3f}]  ({time.time()-t0:.0f} s)")

    T2 = pd.DataFrame(fil)
    print('\n' + T2.to_string(index=False))
    T2.to_csv(f'{OUT}universo_competencia_modelo.csv', index=False)
    print(f'\n  -> {OUT}universo_competencia_modelo.csv')
    print('\nComo se lee. Si la cifra SUBE, quitar el catalogo quito ruido de la poblacion de')
    print('comparacion y el resultado restringido es mas limpio. Si BAJA, el catalogo estaba')
    print('aportando contraste util y hay que declararlo. Las dos lecturas son publicables;')
    print('lo que no seria publicable es correr las dos y quedarse con la que convenga.')
except FileNotFoundError as e:
    print('falta un archivo para construir el universo:', e)

## Paso 3 · El conjunto reducido por bloques

El cuaderno 12 encontró que las 141 variables son 104 bloques de información independiente. Aquí
se evalúa el conjunto reducido con el protocolo canónico, y la regla de elección del
representante se declara **antes** de medir y es mecánica para que no quede margen: el
representante de cada bloque es la variable con mayor cobertura, y si hay empate, la primera en
orden alfabético. Ninguna de las dos condiciones mira la etiqueta.

Lo que se puede concluir y lo que no: si el reducido empata con el completo, la tesis puede decir
que bastan N variables y que el resto es redundancia medida. Lo que **no** se puede hacer es
quedarse con el que salga mejor. El conjunto completo sigue siendo la especificación primaria,
porque está definida por origen y declarada de antemano, y el reducido se reporta como resultado
de robustez.


In [ ]:
# ============ Paso 3 · El conjunto reducido por bloques, con la regla declarada ============
# El cuaderno 12 encontro que las 141 variables son 104 bloques de informacion independiente.
# Aqui se evalua el conjunto reducido con el protocolo canonico. La REGLA de eleccion del
# representante se declara ANTES de medir, y es mecanica para que no haya margen:
#   representante del bloque = la variable con MAYOR cobertura; empate, orden alfabetico.
# Ninguna mira la etiqueta.
UMBRAL_BLOQUE = 0.95

mod = sorted(set(VARS['A']) | set(VARS['B']))
sub = P[mod].sample(min(50000, len(P)), random_state=SEMILLA)
Rk = sub.rank().to_numpy(dtype=float)
Rk = np.where(np.isnan(Rk), np.nanmean(Rk, axis=0), Rk)
Rk = Rk - Rk.mean(0); sd = Rk.std(0); sd[sd == 0] = 1.0
C = np.abs((Rk/sd).T @ (Rk/sd) / len(Rk)); np.fill_diagonal(C, 0.0)

padre = list(range(len(mod)))
def raiz(i):
    while padre[i] != i: padre[i] = padre[padre[i]]; i = padre[i]
    return i
for a, b in zip(*np.where(np.triu(C > UMBRAL_BLOQUE, 1))):
    ra, rb = raiz(int(a)), raiz(int(b))
    if ra != rb: padre[ra] = rb
bloques = {}
for i, c in enumerate(mod): bloques.setdefault(raiz(i), []).append(c)

cob = P[mod].notna().mean()
def representante(bl):
    return sorted(bl, key=lambda c: (-float(cob[c]), c))[0]
REP = {representante(b) for b in bloques.values()}
print(f'bloques: {len(bloques)} de {len(mod)} variables | representantes: {len(REP)}')

A_red = [c for c in VARS['A'] if c in REP]
B_red = [c for c in VARS['B'] if c in REP]
print(f'A: {len(VARS["A"])} -> {len(A_red)} | B: {len(VARS["B"])} -> {len(B_red)}')

fil, t0 = [], time.time()
fil.append(corrida(P, 'y_fantasma', 'proveedor_ruc', A_red, None, etq=f'A reducido ({len(A_red)} var.)'))
print(f"  {fil[-1]['caso']:28s} {fil[-1]['media']:.3f} "
      f"[{fil[-1]['min']:.3f}-{fil[-1]['max']:.3f}]  ({time.time()-t0:.0f} s)")
fil.append(corrida(P[uB], 'y_intragrupo', 'grupo_B', B_red, 5, etq=f'B reducido ({len(B_red)} var.)'))
print(f"  {fil[-1]['caso']:28s} {fil[-1]['media']:.3f} "
      f"[{fil[-1]['min']:.3f}-{fil[-1]['max']:.3f}]  ({time.time()-t0:.0f} s)")

T3 = pd.DataFrame(fil)
print('\n' + T3.to_string(index=False))
json.dump({'A_reducido': A_red, 'B_reducido': B_red, 'umbral': UMBRAL_BLOQUE,
           'regla': 'mayor cobertura; empate, alfabetico'},
          open(f'{OUT}vars_reducidas.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
T3.to_csv(f'{OUT}modelo_reducido.csv', index=False)
print(f'\n  -> {OUT}vars_reducidas.json y {OUT}modelo_reducido.csv')
print('\nQue se puede y que no se puede concluir. Si el reducido empata con el completo, la')
print('tesis puede decir que bastan N variables y que el resto es redundancia medida. Lo que')
print('NO se puede hacer es quedarse con el que salga mejor: el conjunto COMPLETO sigue siendo')
print('la especificacion primaria, porque esta definida por origen y declarada de antemano, y')
print('el reducido se reporta como resultado de robustez. Elegir el mejor de los dos costaria')
print('el optimismo por seleccion que midio F16, que fue de 0,108.')

## Paso 4 · Precisión en el tope

La mediana del percentil responde «dónde caen los positivos en promedio». Una entidad de control
que va a revisar expedientes necesita otra cosa: de los primeros *k* casos de la lista, cuántos
son positivos conocidos y cuánto mejor es eso que sortear al azar.

**Y aquí hay una trampa de interpretación que conviene tener clara antes de ver la tabla.** La
población de comparación son 600 no etiquetados por año, no el panel entero. La proporción de
positivos en esa población es muchísimo mayor que en los 232.382 proveedor-año reales, de modo
que **la columna de precisión no es la que se vería al aplicar el modelo en producción**: ahí
bajaría mucho. Lo que sí es trasladable es el **lift**, porque compara contra el azar dentro de
la misma población. Reportar la precisión como si fuera operativa inflaría el resultado.


In [ ]:
# ============ Paso 4 · Precision en el tope: lo que usaria una entidad de control ============
# La mediana del percentil responde "donde caen los positivos en promedio". Una entidad que va
# a revisar expedientes necesita otra cosa: de los primeros k casos de la lista, cuantos son
# positivos conocidos y cuanto mejor es eso que sortear al azar. Eso es lo que se reporta aqui.
K_TOPE = [10, 25, 50, 100, 250, 500]

def tope(ev, etq):
    ev = ev.dropna(subset=['m']).sort_values('m', ascending=False).reset_index(drop=True)
    n, npos = len(ev), int(ev.y.sum())
    base = npos / n
    fil = []
    for k in [k for k in K_TOPE if k <= n]:
        h = int(ev.y.values[:k].sum())
        fil.append({'caso': etq, 'k': k, 'positivos_en_el_tope': h,
                    'precision_%': round(h / k * 100, 2),
                    'lift': round((h / k) / base, 1) if base > 0 else np.nan,
                    'cobertura_%': round(h / npos * 100, 1) if npos else np.nan})
    return fil

fil, t0 = [], time.time()
for etq, d, cy, cg, conj, nf in [
        ('A fantasma · completo', P,     'y_fantasma',   'proveedor_ruc', 'A', None),
        ('B intragrupo · completo', P[uB], 'y_intragrupo', 'grupo_B',     'B', 5)]:
    cols = VARS[conj]
    y = d[cy].values.astype(int); gi = pd.factorize(d[cg].values)[0]
    X = d[cols].astype(float).values
    ev = evaluacion(X, y, gi, d.anio_proceso.values, nf, K_EV, SEMILLA, Xu=X[y == 0])
    if ev is None: continue
    fil += tope(ev, etq)
    print(f'  {etq}: {int(ev.y.sum())} positivos en {len(ev):,} filas evaluadas '
          f'({time.time()-t0:.0f} s)')

T4 = pd.DataFrame(fil)
print('\n' + T4.to_string(index=False))
T4.to_csv(f'{OUT}precision_tope.csv', index=False)
print(f'\n  -> {OUT}precision_tope.csv')
print('\nComo se lee, y con cuidado. La poblacion de comparacion son 600 no etiquetados por ano,')
print('no el panel entero, asi que la precision de esta tabla NO es la que se veria al aplicar')
print('el modelo a los 232.382 proveedor-ano reales: ahi la proporcion de positivos es mucho')
print('menor y la precision bajaria en consecuencia. Lo que esta tabla sostiene es el LIFT, que')
print('es la comparacion contra el azar dentro de la misma poblacion, y esa si es trasladable.')
print('Reportar la columna de precision como si fuera operativa seria inflar el resultado.')

## Paso 5 · Permutaciones a 199

Con 49 permutaciones el valor p mínimo alcanzable es 1/50 = 0,02, y no hay atajo: un valor p no
puede ser menor que 1/(n+1) por construcción. Para poder afirmar p ≤ 0,005 hacen falta 199
permutaciones, y eso cuesta cuatro veces lo que costó el paso 12.

La celda **cronometra una permutación primero** y te dice cuántos minutos va a tardar el total
antes de empezar. Si es demasiado, baja `N_PERM` y reporta el p que corresponda a ese número: eso
es lo honesto. Lo que no se puede hacer es correr 199, ver que el p no mejora, y reportar el de 49.

Se corre solo para las dos especificaciones principales. Las demás se quedan con el p de 49
permutaciones del cuaderno 8, y el documento declara cuál es cuál.


In [ ]:
# ============ Paso 5 · Permutaciones a 199 (la celda cara) ============
# Con 49 permutaciones el valor p minimo alcanzable es 1/50 = 0,02, y no hay atajo: para poder
# afirmar p <= 0,005 hacen falta 199. Esta celda primero CRONOMETRA una permutacion y te dice
# cuanto va a tardar todo, antes de empezar. Si es mucho, baja N_PERM y reporta el p que
# corresponda a ese numero, que es lo honesto.
N_PERM, K_P = 199, 15

def conglomerados_placebo(tam, pos, r):
    base = tam.drop(index=[c for c in pos if c in tam.index])
    if len(base) == 0: return np.array([], dtype=int)
    libres = base.copy(); sel = []
    for c in pos:
        if len(libres) == 0: libres = base.copy()
        n = int(tam[c]); v = libres.values
        cand = libres.index[v == n]
        if len(cand) == 0:
            dd = np.abs(v - n); cand = libres.index[dd == dd.min()]
        e = r.choice(np.asarray(cand)); sel.append(e)
        if e in libres.index: libres = libres.drop(index=e)
    return np.asarray(sel)

CASOS_P = [('A fantasma · completo', P,     'y_fantasma',   'proveedor_ruc', 'A', None),
           ('B intragrupo · completo', P[uB], 'y_intragrupo', 'grupo_B',     'B', 5)]

filas = []
for nombre, d, cy, cg, conj, nf in CASOS_P:
    cols = VARS[conj]
    y = d[cy].values.astype(int); gi = pd.factorize(d[cg].values)[0]
    an = d.anio_proceso.values;   X = d[cols].astype(float).values
    gp = np.unique(gi[y == 1]); tam = pd.Series(gi).value_counts(); Xu = X[y == 0]
    print(f'\n{nombre}: {y.sum()} positivos en {len(gp)} conglomerados, {len(cols)} variables')

    t0 = time.time(); obs = estadistico(X, y, gi, an, nf, K_P, SEMILLA, Xu=Xu)
    t_obs = time.time() - t0
    print(f'  observado {obs:.3f}  ({t_obs:.0f} s)')
    print(f'  estimado para {N_PERM} permutaciones: {t_obs*N_PERM/60:.0f} minutos')

    nul, t0 = [], time.time()
    for b in range(N_PERM):
        r = np.random.default_rng(SEMILLA + 100000 + b)
        sel = conglomerados_placebo(tam, gp, r)
        yp = np.isin(gi, sel).astype(int)
        nul.append(estadistico(X, yp, gi, an, nf, K_P, SEMILLA + 100000 + b,
                               Xu=X[(y == 0) & (yp == 0)]))
        if (b+1) % 10 == 0:
            print(f'    {b+1}/{N_PERM}  ({time.time()-t0:.0f} s, '
                  f'faltan ~{(time.time()-t0)/(b+1)*(N_PERM-b-1)/60:.0f} min)', end='\r')
    v = np.array([x for x in nul if np.isfinite(x)])
    if len(v) < 10:
        # pasa cuando casi todos los conglomerados son positivos y no hay de donde sacar
        # placebos emparejados por tamano. Es el caso del objetivo B en algunos universos.
        print(f'\n  AVISO: solo {len(v)} permutaciones validas de {N_PERM}. Casi todos los')
        print('  conglomerados son positivos y no hay placebos emparejados. No se reporta p.')
        filas.append({'caso': nombre, 'observado': round(obs, 3), 'n_perm': len(v),
                      'nulo_mediana': np.nan, 'nulo_p2.5': np.nan, 'nulo_p97.5': np.nan,
                      'p_una_cola': np.nan, 'p_minimo_alcanzable': np.nan})
        continue
    p1c = (1 + int((v >= obs).sum())) / (len(v) + 1)
    filas.append({'caso': nombre, 'observado': round(obs, 3), 'n_perm': len(v),
                  'nulo_mediana': round(float(np.median(v)), 3),
                  'nulo_p2.5': round(float(np.percentile(v, 2.5)), 3),
                  'nulo_p97.5': round(float(np.percentile(v, 97.5)), 3),
                  'p_una_cola': round(p1c, 4), 'p_minimo_alcanzable': round(1/(len(v)+1), 4)})
    print(f'\n  nulo {np.median(v):.3f} [{np.percentile(v,2.5):.3f}-{np.percentile(v,97.5):.3f}]'
          f'  ->  p = {p1c:.4f}  (minimo alcanzable con {len(v)}: {1/(len(v)+1):.4f})')

T5 = pd.DataFrame(filas)
print('\n' + T5.to_string(index=False))
T5.to_csv(f'{OUT}permutacion_199.csv', index=False)
print(f'\n  -> {OUT}permutacion_199.csv')
print('\nLa columna p_minimo_alcanzable es la que hay que mirar antes de escribir el numero:')
print('un p no puede ser menor que 1/(n+1), de modo que reportar "p < 0,001" con 199')
print('permutaciones seria imposible por construccion, aunque el observado fuera enorme.')

## Cierre · Qué hacer con estos resultados

**Si el paso 1 dice que el ruido es de bagging**, las cifras finales se vuelven a correr con el K
más alto que el tiempo permita y se declara el cambio de protocolo. Si dice que no, el protocolo
del cuaderno 8 se queda como está y no se toca.

**Los pasos 2 y 3 son resultados que se reportan en los dos sentidos.** Ya están declarados antes
de verlos, de modo que salga lo que salga va al documento. Si alguien del grupo propone correr
una variante más y quedarse con la mejor, la respuesta es F16: eso cuesta 0,108 de optimismo y no
compra nada fuera de muestra.

**El paso 4 es el que convierte el trabajo en algo que una entidad puede usar**, y es también
donde es más fácil exagerar. La frase que va en la tesis es la del lift, no la de la precisión.

**El paso 5 solo cambia un número**, pero es el número que el tribunal va a mirar. Si sale
p = 0,005 en las dos especificaciones principales, la afirmación pasa de «significativo» a
«significativo con el mínimo que permite el diseño». Si no sale, se reporta el que salga.

Lo que ninguno de los cinco pasos cambia es el techo: 15 conglomerados positivos, amplitud 0,442.
Esa cifra es la que manda, y es honesto decir que el trabajo está limitado por el dato y no por
el método.


In [ ]:
# ============ Paso 6 · Permutacion del universo de competencia (solo objetivo A) ============
# Le da su propio nulo a la cifra 0,770. Hay que correr ANTES el paso 2 (crea `comp`).
N_PERM6, K_P = 99, 15

def conglomerados_placebo(tam, pos, r):
    base = tam.drop(index=[c for c in pos if c in tam.index])
    if len(base) == 0: return np.array([], dtype=int)
    libres = base.copy(); sel = []
    for c in pos:
        if len(libres) == 0: libres = base.copy()
        n = int(tam[c]); v = libres.values
        cand = libres.index[v == n]
        if len(cand) == 0:
            dd = np.abs(v - n); cand = libres.index[dd == dd.min()]
        e = r.choice(np.asarray(cand)); sel.append(e)
        if e in libres.index: libres = libres.drop(index=e)
    return np.asarray(sel)

try:
    cols = [c for c in VARS['A'] if c in comp.columns]
    y  = comp['y_fantasma'].values.astype(int)
    gi = pd.factorize(comp['proveedor_ruc'].values)[0]
    an = comp['anio_proceso'].values
    X  = comp[cols].astype(float).values
    gp = np.unique(gi[y == 1]); tam = pd.Series(gi).value_counts(); Xu = X[y == 0]
    print(f'A universo de competencia: {y.sum()} positivos en {len(gp)} conglomerados, {len(cols)} variables')

    t0 = time.time(); obs = estadistico(X, y, gi, an, None, K_P, SEMILLA, Xu=Xu)
    print(f'  observado {obs:.3f}  ({time.time()-t0:.0f} s)')
    print(f'  estimado para {N_PERM6} permutaciones: {(time.time()-t0)*N_PERM6/60:.0f} minutos')

    nul, t0 = [], time.time()
    for b in range(N_PERM6):
        r = np.random.default_rng(SEMILLA + 200000 + b)
        sel = conglomerados_placebo(tam, gp, r)
        yp = np.isin(gi, sel).astype(int)
        nul.append(estadistico(X, yp, gi, an, None, K_P, SEMILLA + 200000 + b,
                               Xu=X[(y == 0) & (yp == 0)]))
        if (b+1) % 10 == 0:
            print(f'    {b+1}/{N_PERM6}  ({time.time()-t0:.0f} s, '
                  f'faltan ~{(time.time()-t0)/(b+1)*(N_PERM6-b-1)/60:.0f} min)', end='\r')
    v = np.array([x for x in nul if np.isfinite(x)])
    p1c = (1 + int((v >= obs).sum())) / (len(v) + 1)
    T6 = pd.DataFrame([{'caso': 'A fantasma · universo de competencia', 'observado': round(obs, 3),
        'n_perm': len(v), 'nulo_mediana': round(float(np.median(v)), 3),
        'nulo_p2.5': round(float(np.percentile(v, 2.5)), 3),
        'nulo_p97.5': round(float(np.percentile(v, 97.5)), 3),
        'p_una_cola': round(p1c, 4), 'p_minimo_alcanzable': round(1/(len(v)+1), 4)}])
    print('\n' + T6.to_string(index=False))
    T6.to_csv(f'{OUT}permutacion_competencia.csv', index=False)
    print(f'\n  -> {OUT}permutacion_competencia.csv')
except NameError as e:
    print('falta correr antes el paso 2 (crea `comp`) o el paso 0:', e)

In [ ]:
# ============ Revisión · ¿por qué cob_peso_rivales == com_grado_completo? ============
# Hace cuatro cosas: (1) confirma que son idénticas, nulos incluidos; (2) mira si sus
# gemelas _pct también lo son; (3) busca en qué cuaderno se construyó cada una y muestra
# esas líneas; (4) compara cada una con las columnas de su propia familia para ver
# a cuál se parece de verdad. No modifica ningún archivo.
import os, json, re

A_, B_ = 'cob_peso_rivales', 'com_grado_completo'

# ---- (1) ¿idénticas? ----
d = P[[A_, B_]]
iguales  = ((d[A_] == d[B_]) | (d[A_].isna() & d[B_].isna())).all()
print(f'(1) idénticas valor por valor (con nulos): {iguales}')
print(f'    nulos: {A_} {d[A_].isna().mean()*100:.1f} %  |  {B_} {d[B_].isna().mean()*100:.1f} %')
print(f'    valores distintos: {d[A_].nunique()}  | mínimo {d[A_].min():.4g}  máximo {d[A_].max():.4g}')

# ---- (2) gemelas _pct y columnas con el mismo contenido ----
print('\n(2) otras columnas del panel idénticas a cualquiera de las dos:')
num = P.select_dtypes('number')
for c in (A_, B_):
    iguales_a = [o for o in num.columns if o != c and
                 ((num[o] == num[c]) | (num[o].isna() & num[c].isna())).all()]
    print(f'    {c}: {iguales_a}')
print('    gemelas _pct:', [c for c in P.columns if c.startswith((A_, B_)) and c.endswith('_pct')])

# ---- (3) dónde se construyeron ----
print('\n(3) dónde aparece cada nombre en tus cuadernos y scripts:')
patron = re.compile(f'{A_}|{B_}')
vistos = 0
for raiz, dirs, archivos in os.walk(R):
    dirs[:] = [x for x in dirs if x not in ('09_historico', '.git', '__pycache__', '.ipynb_checkpoints')]
    for f in archivos:
        ruta = os.path.join(raiz, f)
        try:
            if f.endswith('.ipynb'):
                nb = json.load(open(ruta, encoding='utf-8'))
                for i, cel in enumerate(nb.get('cells', [])):
                    if cel.get('cell_type') != 'code': continue
                    for linea in ''.join(cel.get('source', [])).splitlines():
                        if patron.search(linea):
                            print(f'    {f} · celda {i}: {linea.strip()[:150]}'); vistos += 1
            elif f.endswith('.py'):
                for n, linea in enumerate(open(ruta, encoding='utf-8', errors='ignore'), 1):
                    if patron.search(linea):
                        print(f'    {f} · línea {n}: {linea.strip()[:150]}'); vistos += 1
        except Exception as e:
            pass
    if vistos > 60: print('    (se corta en 60 líneas)'); break
if vistos == 0: print('    no se encontró: la construcción está en un archivo que no es .ipynb ni .py')

# ---- (4) ¿a quién se parece cada una dentro de su familia? ----
print('\n(4) correlación de rango (Spearman) con las columnas de su familia, sobre 20.000 filas:')
m = P.sample(20000, random_state=1)
for c, pref in ((A_, 'cob_'), (B_, 'com_')):
    fam = [o for o in P.select_dtypes('number').columns if o.startswith(pref) and o not in (A_, B_) and not o.endswith('_pct')]
    cr = m[fam].corrwith(m[c], method='spearman').abs().sort_values(ascending=False).head(4)
    print(f'    {c}:  ' + '  '.join(f'{k} {v:.3f}' for k, v in cr.items()))

print('\nCómo leerlo:')
print(' - Si en (3) la misma línea asigna una columna a la otra (algo como df[A] = df[B]) es un error de asignación.')
print(' - Si las dos nacen de cuentas distintas que dan lo mismo por construcción (p. ej. suma de pesos = grado completo),')
print('   entonces es una redundancia legítima y solo hay que documentarla.')
print(' - Pega aquí la salida completa, sobre todo las líneas de (3).')

In [ ]:
# ============ Revisión 2 · ¿misma cuenta por construcción, o error? ============
import os, json, re

# ---- A. prueba con los datos ----
cols = [c for c in ['cob_n_rivales','cob_peso_rivales','cob_n_procesos',
                    'com_grado_total','com_grado_intra','com_grado_completo'] if c in P.columns]
d = P[cols].dropna()
print(f'A. filas con las {len(cols)} columnas completas: {len(d):,}')
print('   son enteros (conteos):', {c: bool((d[c] % 1 == 0).all()) for c in cols})
print(d.describe().T[['min', '50%', 'max']].round(1).to_string())

if {'cob_peso_rivales','cob_n_rivales'} <= set(d.columns):
    r = d.cob_peso_rivales / d.cob_n_rivales
    print(f'\n   peso >= n_rivales en todas las filas: {bool((d.cob_peso_rivales >= d.cob_n_rivales).all())}')
    print(f'   peso / n_rivales (coofertas por rival): mediana {r.median():.2f}, máximo {r.max():.1f}')
if {'com_grado_completo','com_grado_total'} <= set(d.columns):
    print(f'   grado_completo >= grado_total: {(d.com_grado_completo >= d.com_grado_total).mean()*100:.1f} % de las filas')
    print(f'   grado_completo == grado_total: {(d.com_grado_completo == d.com_grado_total).mean()*100:.1f} % de las filas')
if {'com_grado_intra','com_grado_total'} <= set(d.columns):
    print(f'   grado_intra <= grado_total:    {(d.com_grado_intra <= d.com_grado_total).mean()*100:.1f} % de las filas')

# ---- B. búsqueda más amplia, y avisa de lo que no pudo leer ----
patron = re.compile(r'peso_riv|grado_comp|grado_total|grado_intra|peso_rival')
EXT = ('.ipynb', '.py', '.sql', '.txt', '.md')
revisados, fallidos, hits = 0, [], 0
print('\nB. dónde aparecen esos nombres (también en archivos .sql, .txt y .md):')
for raiz, dirs, archivos in os.walk(R):
    dirs[:] = [x for x in dirs if x not in ('09_historico', '.git', '__pycache__', '.ipynb_checkpoints')]
    for f in archivos:
        if not f.endswith(EXT): continue
        ruta = os.path.join(raiz, f)
        try:
            with open(ruta, encoding='utf-8', errors='ignore') as fh:
                if f.endswith('.ipynb'):
                    nb = json.load(fh)
                    for i, cel in enumerate(nb.get('cells', [])):
                        if cel.get('cell_type') != 'code': continue
                        for linea in ''.join(cel.get('source', [])).splitlines():
                            if patron.search(linea) and hits < 50:
                                print(f'   {f} · celda {i}: {linea.strip()[:140]}'); hits += 1
                else:
                    for n, linea in enumerate(fh, 1):
                        if patron.search(linea) and hits < 50:
                            print(f'   {f} · línea {n}: {linea.strip()[:140]}'); hits += 1
            revisados += 1
        except Exception as e:
            fallidos.append(f'{f}  ({type(e).__name__})')
print(f'\n   archivos revisados: {revisados} | no se pudieron abrir: {len(fallidos)}')
for x in fallidos[:15]: print('     -', x)
if hits == 0:
    print('   sin coincidencias: el nombre se arma con código (prefijo + sufijo) o está en otro formato.')
    print('   Dime en qué cuaderno armaste las variables cob_ y com_ (¿el 7?) y busco ahí a mano.')